In [1]:
import pandas as pd
import numpy as np
import json
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


In [2]:
# !pip install spotipy

In [8]:
# JSON 파일 경로
KEY_PATH = "./key_folder/spotify_api_key.json"
# JSON 파일 읽어서 키 로드
with open(KEY_PATH, "r") as f:
    key_data = json.load(f)
# 원하는 키 추출 (json 내부 구조에 맞춰 수정 가능)
client_id = key_data["client_id"]
client_secret = key_data["client_secret"]

auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)


In [3]:
artist = sp.search(q="seventeen", type="artist", limit=1)['artists']['items'][0]
print(artist['name'], artist['followers']['total'], artist['popularity'])

SEVENTEEN 13628171 81


In [4]:
# -------------------
# 1. 아티스트 정보
# -------------------
artist_name = "seventeen"
result = sp.search(q=artist_name, type="artist", limit=1)
artist = result['artists']['items'][0]

print("이름:", artist['name'])
print("팔로워:", artist['followers']['total'])
print("장르:", artist['genres'])
print("인기도:", artist['popularity'])


이름: SEVENTEEN
팔로워: 13628171
장르: ['k-pop']
인기도: 81


- popularity
    - 0~100 범위의 정수
	- 높을수록 인기 있음을 의미
	- Spotify에서 최근 몇 주간 스트리밍 횟수, 청취자 수, 재생 추세 등을 기반으로 계산됨
	- 단순 재생 수가 아닌 상대적 순위 지표

- 트랙 popularity : 최근 재생량 + 청취자 수 + 저장 횟수 등 반영
- 아티스트 popularity : 아티스트 전체 트랙 인기 + 팔로워 수 + 청취자 수 반영
- 앨범 popularity : 앨범 내 트랙들의 인기 + 발매 후 청취 추세 반영
- 상대적 지표 : 동일 국가/지역에서 상대적 순위, 절대 재생 수 아님

In [5]:
# -------------------
# 2. 인기 트랙
# -------------------
top_tracks = sp.artist_top_tracks(artist['id'], country='KR')
track_data = []
for track in top_tracks['tracks']:
    track_data.append({
        "track_name": track['name'],
        "album_name": track['album']['name'],
        "release_date": track['album']['release_date'],
        "popularity": track['popularity'],
        "preview_url": track['preview_url'],
        "artists": [a['name'] for a in track['artists']]
    })

df_tracks = pd.DataFrame(track_data)
df_tracks.to_csv("seventeen_top_tracks.csv", index=False)
print(df_tracks)

                            track_name  \
0                              THUNDER   
1  LOVE, MONEY, FAME (feat. DJ Khaled)   
2                                Super   
3                                  HOT   
4                        Rock with you   
5                            VERY NICE   
6                             Darl+ing   
7                              MAESTRO   
8                      Don't Wanna Cry   
9                             Pretty U   

                                          album_name release_date  popularity  \
0               SEVENTEEN 5th Album 'HAPPY BURSTDAY'   2025-05-26          78   
1  LOVE, MONEY, FAME (feat. DJ Khaled) (Timbaland...   2024-11-01          64   
2            SEVENTEEN BEST ALBUM '17 IS RIGHT HERE'   2024-04-29          67   
3                 SEVENTEEN 4th Album 'Face the Sun'   2022-05-27          70   
4                 SEVENTEEN 9th Mini Album 'Attacca'   2021-10-22          68   
5                        Love&Letter repackage albu

In [6]:

# -------------------
# 0. 인증 (Client ID / Secret 필요)
# -------------------
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id="_",
    client_secret="_"
))

# -------------------
# 1. 아티스트 검색 & 정보 가져오기
# -------------------
artist_name = "Seventeen"  # 원하는 아티스트명
result = sp.search(q=artist_name, type="artist", limit=1)
artist = result['artists']['items'][0]

artist_info = {
    "artist_name": artist['name'],
    "genres": artist['genres'],
    "popularity": artist['popularity'],
    "followers_total": artist['followers']['total']
}

# -------------------
# 2. 인기 트랙 가져오기
# -------------------
top_tracks = sp.artist_top_tracks(artist['id'], country='KR')

track_data = []
for track in top_tracks['tracks']:
    # 기본 트랙 정보
    track_info = {
        "track_name": track['name'],
        "album_name": track['album']['name'],
        "release_date": track['album']['release_date'],
        "duration_ms": track['duration_ms'],
        "explicit": track['explicit'],
        "available_markets": track['available_markets'],
        "is_playable": track.get('is_playable', None),
        "popularity": track['popularity'],
    }

    track_data.append(track_info)

# -------------------
# 3. DataFrame으로 정리
# -------------------
df_tracks = pd.DataFrame(track_data)

print("아티스트 정보:")
print(artist_info)
print("\n트랙별 정보:")
print(df_tracks.head())

아티스트 정보:
{'artist_name': 'SEVENTEEN', 'genres': ['k-pop'], 'popularity': 81, 'followers_total': 13628171}

트랙별 정보:
                            track_name  \
0                              THUNDER   
1  LOVE, MONEY, FAME (feat. DJ Khaled)   
2                                Super   
3                                  HOT   
4                        Rock with you   

                                          album_name release_date  \
0               SEVENTEEN 5th Album 'HAPPY BURSTDAY'   2025-05-26   
1  LOVE, MONEY, FAME (feat. DJ Khaled) (Timbaland...   2024-11-01   
2            SEVENTEEN BEST ALBUM '17 IS RIGHT HERE'   2024-04-29   
3                 SEVENTEEN 4th Album 'Face the Sun'   2022-05-27   
4                 SEVENTEEN 9th Mini Album 'Attacca'   2021-10-22   

   duration_ms  explicit                                  available_markets  \
0       164040     False  [AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...   
1       186386     False  [AR, AU, AT, BE, BO, BR, BG, CA, 